In [16]:
import pandas as pd
import numpy as np
import sys
import os

# Add parent directory to path to import feature engineering utilities
sys.path.append('..')
from feature_engineering_utils import create_all_features, get_all_feature_columns

# Feature Engineering & Model Testing - Scraped Data

This notebook:
1. Loads the processed user activity data
2. Applies feature engineering
3. Tests pre-trained models from the relative dataset

**Data source:** Firehose API crawl (Dec 31 - Jan 16, 2026)

**Models tested:** Week 2 predictions (days 7-13)

## 1. Load and Engineer Features

In [17]:
# Load processed user activity data
user_activity_path = "../../data/ale_simplicistic_model/scraped/processed/user_activity.parquet"
df = pd.read_parquet(user_activity_path)

print(f"Loaded {len(df):,} users")
print(f"Columns: {list(df.columns)}")
print(f"\nSample data:")
df.head()

Loaded 1,081 users
Columns: ['did_id', 'join_date', 'posts_vec', 'blocks_actor_vec', 'blocks_subject_vec', 'follows_actor_vec', 'follows_subject_vec', 'likes_actor_vec', 'likes_subject_vec']

Sample data:


,did_id,join_date,posts_vec,blocks_actor_vec,blocks_subject_vec,follows_actor_vec,follows_subject_vec,likes_actor_vec,likes_subject_vec
0,414918,2026-01-08,"[2, 1, 0, 5, 2, 0, 0]","[0, 1, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 1, 0, 0]","[0, 0, 0, 1, 0, 2, 0]","[4, 5, 2, 0, 0, 1, 0]","[0, 0, 0, 0, 0, 1, 0]"
1,2438083,2026-01-02,"[11, 6, 14, 7, 23, 1, 22]","[0, 1, 1, 0, 0, 0, 1]","[1, 1, 2, 0, 0, 0, 0]","[26, 1, 1, 0, 6, 1, 12]","[2, 0, 0, 0, 1, 0, 2]","[12, 6, 12, 1, 12, 1, 19]","[0, 0, 0, 0, 0, 0, 0]"
2,185592,2026-01-08,"[4, 4, 4, 7, 1, 5, 4]","[0, 1, 0, 0, 0, 0, 0]","[2, 0, 0, 0, 0, 0, 1]","[0, 0, 2, 0, 1, 2, 1]","[0, 0, 0, 0, 0, 1, 1]","[1, 2, 4, 5, 0, 2, 1]","[0, 0, 0, 0, 0, 0, 0]"
3,591427,2026-01-04,"[2, 0, 5, 2, 5, 5, 2]","[1, 1, 0, 0, 1, 0, 1]","[0, 0, 0, 0, 1, 0, 1]","[1, 0, 1, 0, 12, 0, 13]","[0, 0, 1, 0, 1, 0, 4]","[12, 13, 11, 16, 13, 8, 19]","[0, 0, 0, 0, 0, 0, 0]"
4,2361852,2026-01-04,"[96, 17, 65, 38, 34, 44, 19]","[5, 0, 4, 1, 1, 0, 1]","[2, 0, 0, 0, 0, 0, 0]","[14, 0, 1, 0, 0, 1, 0]","[5, 1, 1, 0, 0, 0, 0]","[185, 88, 59, 59, 32, 83, 37]","[0, 0, 0, 0, 0, 0, 0]"


In [18]:
# Apply feature engineering
print("Creating features...")
df_features = create_all_features(df)

print(f"\nFeatures created: {len(get_all_feature_columns())} features")
print(f"Total columns: {len(df_features.columns)}")
df_features.head()

Creating features...

Features created: 30 features
Total columns: 39


,did_id,join_date,posts_vec,blocks_actor_vec,blocks_subject_vec,follows_actor_vec,follows_subject_vec,likes_actor_vec,likes_subject_vec,posts_total,...,blocks_to_follows_ratio,posts_first_active_day,posts_last_active_day,follows_made_first_day,follows_made_last_day,likes_made_first_day,likes_made_last_day,blocks_initiated_first_day,blocks_initiated_last_day,last_active_overall
0,414918,2026-01-08,"[2, 1, 0, 5, 2, 0, 0]","[0, 1, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 1, 0, 0]","[0, 0, 0, 1, 0, 2, 0]","[4, 5, 2, 0, 0, 1, 0]","[0, 0, 0, 0, 0, 1, 0]",10,...,1.000000,0,4,4,4,0,5,1,1,5
1,2438083,2026-01-02,"[11, 6, 14, 7, 23, 1, 22]","[0, 1, 1, 0, 0, 0, 1]","[1, 1, 2, 0, 0, 0, 0]","[26, 1, 1, 0, 6, 1, 12]","[2, 0, 0, 0, 1, 0, 2]","[12, 6, 12, 1, 12, 1, 19]","[0, 0, 0, 0, 0, 0, 0]",84,...,0.063830,0,6,0,6,0,6,1,6,6
2,185592,2026-01-08,"[4, 4, 4, 7, 1, 5, 4]","[0, 1, 0, 0, 0, 0, 0]","[2, 0, 0, 0, 0, 0, 1]","[0, 0, 2, 0, 1, 2, 1]","[0, 0, 0, 0, 0, 1, 1]","[1, 2, 4, 5, 0, 2, 1]","[0, 0, 0, 0, 0, 0, 0]",29,...,0.166667,0,6,2,6,0,6,1,1,6
3,591427,2026-01-04,"[2, 0, 5, 2, 5, 5, 2]","[1, 1, 0, 0, 1, 0, 1]","[0, 0, 0, 0, 1, 0, 1]","[1, 0, 1, 0, 12, 0, 13]","[0, 0, 1, 0, 1, 0, 4]","[12, 13, 11, 16, 13, 8, 19]","[0, 0, 0, 0, 0, 0, 0]",21,...,0.148148,0,6,0,6,0,6,0,6,6
4,2361852,2026-01-04,"[96, 17, 65, 38, 34, 44, 19]","[5, 0, 4, 1, 1, 0, 1]","[2, 0, 0, 0, 0, 0, 0]","[14, 0, 1, 0, 0, 1, 0]","[5, 1, 1, 0, 0, 0, 0]","[185, 88, 59, 59, 32, 83, 37]","[0, 0, 0, 0, 0, 0, 0]",313,...,0.750000,0,6,0,5,0,6,0,6,6


In [19]:
# Get feature matrix
feature_cols = get_all_feature_columns()
X_test = df_features[feature_cols]

print(f"Feature matrix shape: {X_test.shape}")
print(f"\nFeature statistics:")
X_test.describe()

Feature matrix shape: (1081, 30)

Feature statistics:


,posts_total,posts_avg,posts_active_days,posts_day0,blocks_initiated_total,blocks_received_total,blocks_initiated_active_days,blocks_received_active_days,follows_made_total,follows_received_total,...,blocks_to_follows_ratio,posts_first_active_day,posts_last_active_day,follows_made_first_day,follows_made_last_day,likes_made_first_day,likes_made_last_day,blocks_initiated_first_day,blocks_initiated_last_day,last_active_overall
count,1081.000000,1081.000000,1081.000000,1081.000000,1081.000000,1081.000000,1081.000000,1081.000000,1081.000000,1081.000000,...,1081.000000,1081.000000,1081.000000,1081.000000,1081.000000,1081.000000,1081.000000,1081.000000,1081.000000,1081.000000
mean,15.546716,2.220959,2.048104,3.160037,4.067530,0.561517,1.493062,0.286772,5.131360,2.844588,...,3.244516,0.151711,2.177613,-0.180389,0.471785,0.187789,2.950046,1.498612,2.360777,3.965772
std,49.037702,7.005386,2.209302,9.956912,44.270717,2.849652,1.053301,0.810293,41.070528,13.227633,...,44.201595,1.432144,2.921488,1.458635,2.318083,1.251991,2.867975,1.798726,2.140720,2.220221
min,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,...,0.000923,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,...,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,2.000000
50%,2.000000,0.285714,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,...,1.000000,0.000000,2.000000,-1.000000,-1.000000,0.000000,4.000000,1.000000,2.000000,5.000000
75%,8.000000,1.142857,3.000000,2.000000,2.000000,0.000000,2.000000,0.000000,1.000000,1.000000,...,2.000000,0.000000,5.000000,0.000000,1.000000,0.000000,6.000000,3.000000,4.000000,6.000000
max,371.000000,53.000000,7.000000,96.000000,1449.000000,64.000000,7.000000,6.000000,1083.000000,228.000000,...,1449.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000


## 2. Load Pre-trained Models

In [20]:
import joblib
import os

models_dir = "../../data/ale_simplicistic_model/relative/model_ready"

# Hardcoded model configurations to test
# Set which model set to use: 'regular' or 'balanced'
MODEL_SET = 'regular'  # Change to 'regular' to test regular models

if MODEL_SET == 'regular':
    timestamp = '20260128_114227'
    model_configs = {
        'logisticregression': f'logisticregression_week2_{timestamp}.pkl',
        'randomforest': f'randomforest_week2_{timestamp}.pkl',
        'gradientboosting': f'gradientboosting_week2_{timestamp}.pkl'
    }
    scaler_file = f'scaler_week2_{timestamp}.pkl'
    metadata_file = f'metadata_week2_{timestamp}.pkl'
    print(f"Loading REGULAR models (timestamp: {timestamp})")
    
elif MODEL_SET == 'balanced':
    timestamp = '20260128_124913'  # Using the later timestamp for balanced
    model_configs = {
        'logisticregression': f'balanced/logisticregression_week2balanced_{timestamp}.pkl',
        'randomforest': f'balanced/randomforest_week2balanced_{timestamp}.pkl',
        'gradientboosting': f'balanced/gradientboosting_week2balanced_{timestamp}.pkl'
    }
    # Note: The naming pattern is week2balanced, not week2_balanced
    scaler_file = f'balanced/scaler_week2balanced_{timestamp}.pkl'
    metadata_file = f'balanced/metadata_week2balanced_{timestamp}.pkl'
    print(f"Loading BALANCED models (timestamp: {timestamp})")

# Load models with error handling
week = 'week2'
loaded_models = {week: {'models': {}}}

print("\n" + "="*60)
print(f"Model directory: {models_dir}")
print("="*60)

# Load scaler using joblib (as saved in model_utils.py)
scaler_path = os.path.join(models_dir, scaler_file)
print(f"\n📊 Loading scaler from: {scaler_file}")
try:
    loaded_models[week]['scaler'] = joblib.load(scaler_path)
    print(f"✓ Loaded scaler successfully")
except Exception as e:
    print(f"✗ Failed to load scaler: {type(e).__name__}: {e}")

# Load metadata using joblib (as saved in model_utils.py)
metadata_path = os.path.join(models_dir, metadata_file)
print(f"\n📋 Loading metadata from: {metadata_file}")
print(f"   Full path: {metadata_path}")
print(f"   File exists: {os.path.exists(metadata_path)}")

try:
    loaded_models[week]['metadata'] = joblib.load(metadata_path)
    print(f"✓ Loaded metadata successfully")
    
    # Metadata structure from model_utils.py uses 'feature_columns', not 'selected_features'
    if 'feature_columns' in loaded_models[week]['metadata']:
        feature_cols = loaded_models[week]['metadata']['feature_columns']
        print(f"  - Feature columns: {len(feature_cols)}")
    elif 'selected_features' in loaded_models[week]['metadata']:
        feature_cols = loaded_models[week]['metadata']['selected_features']
        print(f"  - Selected features: {len(feature_cols)}")
    else:
        print(f"  - Metadata keys: {list(loaded_models[week]['metadata'].keys())}")
        
except Exception as e:
    print(f"✗ Failed to load metadata: {type(e).__name__}: {e}")

# Load each model using joblib (as saved in model_utils.py)
print("\n" + "="*60)
print("🤖 Loading models...")
print("="*60)

for model_name, model_file in model_configs.items():
    model_path = os.path.join(models_dir, model_file)
    print(f"\n  Loading {model_name}...")
    print(f"    Path: {model_file}")
    
    if os.path.exists(model_path):
        try:
            loaded_models[week]['models'][model_name] = joblib.load(model_path)
            print(f"    ✓ Success")
        except Exception as e:
            print(f"    ✗ Failed: {type(e).__name__}: {e}")
    else:
        print(f"    ✗ File not found")

print("\n" + "="*60)
print("📦 SUMMARY")
print("="*60)
print(f"✓ Loaded {len(loaded_models[week]['models'])} models: {list(loaded_models[week]['models'].keys())}")
print(f"✓ Scaler: {'✓' if 'scaler' in loaded_models[week] else '✗'}")
print(f"✓ Metadata: {'✓' if 'metadata' in loaded_models[week] else '✗'}")
print("="*60)

Loading REGULAR models (timestamp: 20260128_114227)

Model directory: ../../data/ale_simplicistic_model/relative/model_ready

📊 Loading scaler from: scaler_week2_20260128_114227.pkl
✓ Loaded scaler successfully

📋 Loading metadata from: metadata_week2_20260128_114227.pkl
   Full path: ../../data/ale_simplicistic_model/relative/model_ready/metadata_week2_20260128_114227.pkl
   File exists: True
✓ Loaded metadata successfully
  - Feature columns: 22

🤖 Loading models...

  Loading logisticregression...
    Path: logisticregression_week2_20260128_114227.pkl
    ✓ Success

  Loading randomforest...
    Path: randomforest_week2_20260128_114227.pkl
    ✓ Success

  Loading gradientboosting...
    Path: gradientboosting_week2_20260128_114227.pkl
    ✓ Success

📦 SUMMARY
✓ Loaded 3 models: ['logisticregression', 'randomforest', 'gradientboosting']
✓ Scaler: ✓
✓ Metadata: ✓


## 3. Load Ground Truth (Week 2 Blocks)

We need to load the actual blocks that happened in week 2 (days 7-13) for our test users.

In [21]:
import duckdb

# Load blocks from scraped data
blocks_path = "../../data/scraped/cleaned/blocks.parquet"
profiles_path = "../../data/scraped/cleaned/profiles.parquet"

con = duckdb.connect()

# Get test user IDs
test_users = df_features['did_id'].tolist()
test_users_df = pd.DataFrame({'did_id': test_users})
con.register('test_users', test_users_df)

print(f"Loading ground truth for {len(test_users):,} test users...")

# Query to get block COUNTS for week 2 (days 7-13)
query = f"""
WITH blocks_with_days AS (
    SELECT 
        b.did_id,
        b.created_date AS block_date,
        p.created_date AS join_date,
        DATE_DIFF('day', CAST(p.created_date AS TIMESTAMP), CAST(b.created_date AS TIMESTAMP)) AS days_since_join
    FROM read_parquet('{blocks_path}') b
    INNER JOIN read_parquet('{profiles_path}') p ON b.did_id = p.did_id
    INNER JOIN test_users tu ON b.did_id = tu.did_id
    WHERE DATE_DIFF('day', CAST(p.created_date AS TIMESTAMP), CAST(b.created_date AS TIMESTAMP)) BETWEEN 7 AND 13
)
SELECT 
    did_id,
    COUNT(*) as block_count
FROM blocks_with_days
GROUP BY did_id
"""

week2_blocks = con.execute(query).df()
print(f"Week 2 (days 7-13): {len(week2_blocks):,} users blocked someone")

# Create MULTICLASS labels (matching training data)
# Class 0: 0 blocks (Inactive)
# Class 1: 1-3 blocks (Low Activity)
# Class 2: 4+ blocks (High Activity)

# Create a dataframe with all test users
y_true_df = pd.DataFrame({'did_id': test_users})

# Merge with block counts (users not in week2_blocks have 0 blocks)
y_true_df = y_true_df.merge(week2_blocks, on='did_id', how='left')
y_true_df['block_count'] = y_true_df['block_count'].fillna(0).astype(int)

# Create multiclass labels using the same bucketing as training
def create_block_class(count):
    """Convert block count to class label"""
    if count == 0:
        return 0  # Inactive
    elif count <= 3:
        return 1  # Low Activity (1-3 blocks)
    else:
        return 2  # High Activity (4+ blocks)

y_true_df['block_class'] = y_true_df['block_count'].apply(create_block_class)
y_true = y_true_df['block_class'].values

print(f"\nWeek 2 multiclass labels:")
print(f"  Class 0 (Inactive, 0 blocks):       {(y_true == 0).sum():>6,} ({(y_true == 0).sum()/len(y_true)*100:>5.2f}%)")
print(f"  Class 1 (Low Activity, 1-3 blocks): {(y_true == 1).sum():>6,} ({(y_true == 1).sum()/len(y_true)*100:>5.2f}%)")
print(f"  Class 2 (High Activity, 4+ blocks): {(y_true == 2).sum():>6,} ({(y_true == 2).sum()/len(y_true)*100:>5.2f}%)")
print(f"  Total:                               {len(y_true):>6,}")

# Show some statistics
print(f"\nBlock count distribution for active users:")
active_users = y_true_df[y_true_df['block_count'] > 0]
if len(active_users) > 0:
    print(f"  Min blocks:    {active_users['block_count'].min()}")
    print(f"  Max blocks:    {active_users['block_count'].max()}")
    print(f"  Mean blocks:   {active_users['block_count'].mean():.2f}")
    print(f"  Median blocks: {active_users['block_count'].median():.0f}")

con.close()

Loading ground truth for 1,081 test users...
Week 2 (days 7-13): 135 users blocked someone

Week 2 multiclass labels:
  Class 0 (Inactive, 0 blocks):          902 (83.44%)
  Class 1 (Low Activity, 1-3 blocks):    104 ( 9.62%)
  Class 2 (High Activity, 4+ blocks):     75 ( 6.94%)
  Total:                                1,081

Block count distribution for active users:
  Min blocks:    1
  Max blocks:    251
  Mean blocks:   12.83
  Median blocks: 2


## 4. Make Predictions and Evaluate

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# Store results
results = []

print(f"\n{'='*80}")
print(f"TESTING WEEK 2 MODELS (MULTICLASS)")
print(f"{'='*80}")

# Get scaler and features from metadata
scaler = loaded_models[week]['scaler']
metadata = loaded_models[week]['metadata']

# Metadata uses 'feature_columns' not 'selected_features' (from model_utils.py)
if 'feature_columns' in metadata:
    feature_cols_to_use = metadata['feature_columns']
elif 'selected_features' in metadata:
    feature_cols_to_use = metadata['selected_features']
else:
    raise KeyError(f"Metadata doesn't contain 'feature_columns' or 'selected_features'. Keys: {list(metadata.keys())}")

print(f"\nUsing {len(feature_cols_to_use)} features")
print(f"Test set size: {len(X_test):,} users")
print(f"Classes: {metadata.get('classes', [0, 1, 2])}")

# Prepare test data with the correct features
X_test_selected = X_test[feature_cols_to_use]
X_test_scaled = scaler.transform(X_test_selected)

for model_name, model in loaded_models[week]['models'].items():
    print(f"\n{'='*70}")
    print(f"Model: {model_name.upper()}")
    print(f"{'='*70}")
    
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled) if hasattr(model, 'predict_proba') else None
    
    # Calculate multiclass metrics
    accuracy = accuracy_score(y_true, y_pred)
    
    # Multiclass metrics with 'macro' and 'weighted' averaging
    precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    precision_weighted = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall_weighted = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    print(f"\nOverall Metrics:")
    print(f"  Accuracy:          {accuracy:.4f}")
    print(f"\nMacro-averaged (equal weight per class):")
    print(f"  Precision (macro): {precision_macro:.4f}")
    print(f"  Recall (macro):    {recall_macro:.4f}")
    print(f"  F1-Score (macro):  {f1_macro:.4f}")
    print(f"\nWeighted-averaged (weighted by support):")
    print(f"  Precision (wtd):   {precision_weighted:.4f}")
    print(f"  Recall (wtd):      {recall_weighted:.4f}")
    print(f"  F1-Score (wtd):    {f1_weighted:.4f}")
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nConfusion Matrix:")
    print(f"                  Predicted")
    print(f"              Class 0  Class 1  Class 2")
    print(f"  Actual  0:  {cm[0,0]:>7,}  {cm[0,1]:>7,}  {cm[0,2]:>7,}")
    print(f"          1:  {cm[1,0]:>7,}  {cm[1,1]:>7,}  {cm[1,2]:>7,}")
    print(f"          2:  {cm[2,0]:>7,}  {cm[2,1]:>7,}  {cm[2,2]:>7,}")
    
    print(f"\nClass Labels:")
    print(f"  Class 0: Inactive (0 blocks)")
    print(f"  Class 1: Low Activity (1-3 blocks)")
    print(f"  Class 2: High Activity (4+ blocks)")
    
    # Per-class metrics
    print(f"\nPer-Class Metrics:")
    precision_per_class = precision_score(y_true, y_pred, average=None, zero_division=0)
    recall_per_class = recall_score(y_true, y_pred, average=None, zero_division=0)
    f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
    
    for i, (p, r, f) in enumerate(zip(precision_per_class, recall_per_class, f1_per_class)):
        class_name = ['Inactive', 'Low Activity', 'High Activity'][i]
        print(f"  Class {i} ({class_name:13s}): Precision={p:.4f}, Recall={r:.4f}, F1={f:.4f}")
    
    # Store results
    results.append({
        'model': model_name,
        'accuracy': accuracy,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'f1_weighted': f1_weighted,
        'cm_00': int(cm[0,0]), 'cm_01': int(cm[0,1]), 'cm_02': int(cm[0,2]),
        'cm_10': int(cm[1,0]), 'cm_11': int(cm[1,1]), 'cm_12': int(cm[1,2]),
        'cm_20': int(cm[2,0]), 'cm_21': int(cm[2,1]), 'cm_22': int(cm[2,2])
    })

# Create results dataframe
results_df = pd.DataFrame(results)
print(f"\n{'='*80}")
print("SUMMARY OF RESULTS")
print(f"{'='*80}\n")
print(results_df[['model', 'accuracy', 'f1_macro', 'f1_weighted', 
                   'precision_macro', 'recall_macro']].to_string(index=False))


TESTING WEEK 2 MODELS (MULTICLASS)

Using 22 features
Test set size: 1,081 users
Classes: [0, 1, 2]

Model: LOGISTICREGRESSION

Overall Metrics:
  Accuracy:          0.8316

Macro-averaged (equal weight per class):
  Precision (macro): 0.5462
  Recall (macro):    0.5154
  F1-Score (macro):  0.5237

Weighted-averaged (weighted by support):
  Precision (wtd):   0.7952
  Recall (wtd):      0.8316
  F1-Score (wtd):    0.8107

Confusion Matrix:
                  Predicted
              Class 0  Class 1  Class 2
  Actual  0:      851       26       25
          1:       88       10        6
          2:       24       13       38

Class Labels:
  Class 0: Inactive (0 blocks)
  Class 1: Low Activity (1-3 blocks)
  Class 2: High Activity (4+ blocks)

Per-Class Metrics:
  Class 0 (Inactive     ): Precision=0.8837, Recall=0.9435, F1=0.9126
  Class 1 (Low Activity ): Precision=0.2041, Recall=0.0962, F1=0.1307
  Class 2 (High Activity): Precision=0.5507, Recall=0.5067, F1=0.5278

Model: RANDOMFOR

/home/ale/micromamba/envs/bluesky/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/ale/micromamba/envs/bluesky/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/ale/micromamba/envs/bluesky/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(
/home/ale/micromamba/envs/bluesky/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(


## 5. Save Results

In [23]:
# Save results
results_path = "../../data/ale_simplicistic_model/scraped/model_results.parquet"
os.makedirs(os.path.dirname(results_path), exist_ok=True)
results_df.to_parquet(results_path, index=False)

print(f"Results saved to: {results_path}")

# Also save as CSV for easy viewing
csv_path = results_path.replace('.parquet', '.csv')
results_df.to_csv(csv_path, index=False)
print(f"Results also saved to: {csv_path}")

Results saved to: ../../data/ale_simplicistic_model/scraped/model_results.parquet
Results also saved to: ../../data/ale_simplicistic_model/scraped/model_results.csv
